# Exploratory Data Analysis (EDA) - Emotion Dataset
## 20 Emotions Classification Dataset

This notebook performs comprehensive EDA on the emotion dataset containing text samples labeled with 20 different emotions.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/emotion_dataset_v5_clean.csv')

print(f"Dataset shape: {df.shape}")
print(f"Number of samples: {len(df):,}")
print(f"Number of features: {df.shape[1]}")

## 2. Basic Data Overview

In [ ]:
# Display first few rows
print("First 10 rows of the dataset:")
df.head(10)

In [ ]:
# Dataset info
print("Dataset Information:")
df.info()

In [ ]:
# Column names and data types
print("\nColumn Names and Data Types:")
print(df.dtypes)

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df.describe(include='all')

## 3. Missing Values Analysis

In [ ]:
# Check for missing values
print("Missing Values:")
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Percentage': missing_percentage.values
})
print(missing_df)

# Visualize missing values
if missing_values.sum() > 0:
    plt.figure(figsize=(10, 4))
    missing_values[missing_values > 0].plot(kind='bar')
    plt.title('Missing Values by Column')
    plt.ylabel('Count')
    plt.xlabel('Columns')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("\n✓ No missing values found in the dataset!")

## 4. Emotion Distribution Analysis

In [ ]:
# Count of each emotion
emotion_counts = df['emotion'].value_counts()
print("Emotion Distribution:")
print(emotion_counts)
print(f"\nNumber of unique emotions: {df['emotion'].nunique()}")

In [ ]:
# Visualize emotion distribution - Bar plot
plt.figure(figsize=(14, 6))
emotion_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Distribution of Emotions in Dataset', fontsize=16, fontweight='bold')
plt.xlabel('Emotion', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart for emotion distribution
plt.figure(figsize=(12, 12))
colors = sns.color_palette('husl', len(emotion_counts))
plt.pie(emotion_counts.values, labels=emotion_counts.index, autopct='%1.1f%%', 
        startangle=90, colors=colors)
plt.title('Emotion Distribution (Percentage)', fontsize=16, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Check for class imbalance
print("\nClass Balance Analysis:")
print(f"Most common emotion: {emotion_counts.index[0]} ({emotion_counts.values[0]:,} samples)")
print(f"Least common emotion: {emotion_counts.index[-1]} ({emotion_counts.values[-1]:,} samples)")
print(f"Imbalance ratio: {emotion_counts.values[0] / emotion_counts.values[-1]:.2f}x")

# Calculate percentage for each emotion
emotion_percentage = (emotion_counts / len(df) * 100).round(2)
balance_df = pd.DataFrame({
    'Emotion': emotion_counts.index,
    'Count': emotion_counts.values,
    'Percentage': emotion_percentage.values
})
print("\n", balance_df)

## 5. Text Length Analysis

In [ ]:
# Assuming the text column is named 'text', 'sentence', or similar
# Let's first check column names
print("Column names:")
print(df.columns.tolist())

In [ ]:
# Determine the text column (usually 'text', 'sentence', 'content', etc.)
text_columns = [col for col in df.columns if col.lower() in ['text', 'sentence', 'content', 'message', 'tweet']]
if text_columns:
    text_col = text_columns[0]
else:
    # If not found, use the column that's not 'emotion'
    text_col = [col for col in df.columns if col != 'emotion'][0]

print(f"Text column identified: '{text_col}'")

In [ ]:
# Calculate text statistics
df['text_length'] = df[text_col].astype(str).apply(len)
df['word_count'] = df[text_col].astype(str).apply(lambda x: len(x.split()))
df['avg_word_length'] = df[text_col].astype(str).apply(lambda x: np.mean([len(word) for word in x.split()] if x.split() else [0]))

print("Text Statistics:")
print(f"Average text length: {df['text_length'].mean():.2f} characters")
print(f"Average word count: {df['word_count'].mean():.2f} words")
print(f"Average word length: {df['avg_word_length'].mean():.2f} characters")
print(f"\nMin text length: {df['text_length'].min()} characters")
print(f"Max text length: {df['text_length'].max()} characters")
print(f"Min word count: {df['word_count'].min()} words")
print(f"Max word count: {df['word_count'].max()} words")

In [ ]:
# Distribution of text lengths
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Text length distribution
axes[0, 0].hist(df['text_length'], bins=50, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Text Length (Characters)', fontweight='bold')
axes[0, 0].set_xlabel('Text Length')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(alpha=0.3)

# Word count distribution
axes[0, 1].hist(df['word_count'], bins=50, color='coral', edgecolor='black')
axes[0, 1].set_title('Distribution of Word Count', fontweight='bold')
axes[0, 1].set_xlabel('Word Count')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(alpha=0.3)

# Box plot for text length
axes[1, 0].boxplot(df['text_length'], vert=True)
axes[1, 0].set_title('Text Length Box Plot', fontweight='bold')
axes[1, 0].set_ylabel('Text Length')
axes[1, 0].grid(alpha=0.3)

# Box plot for word count
axes[1, 1].boxplot(df['word_count'], vert=True)
axes[1, 1].set_title('Word Count Box Plot', fontweight='bold')
axes[1, 1].set_ylabel('Word Count')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Text length by emotion
emotion_text_stats = df.groupby('emotion').agg({
    'text_length': ['mean', 'median', 'std'],
    'word_count': ['mean', 'median', 'std']
}).round(2)

print("Text Statistics by Emotion:")
print(emotion_text_stats)

In [ ]:
# Visualize average text length and word count by emotion
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Average text length by emotion
avg_length_by_emotion = df.groupby('emotion')['text_length'].mean().sort_values(ascending=False)
avg_length_by_emotion.plot(kind='bar', ax=axes[0], color='teal', edgecolor='black')
axes[0].set_title('Average Text Length by Emotion', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Emotion', fontsize=12)
axes[0].set_ylabel('Average Text Length', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Average word count by emotion
avg_words_by_emotion = df.groupby('emotion')['word_count'].mean().sort_values(ascending=False)
avg_words_by_emotion.plot(kind='bar', ax=axes[1], color='orange', edgecolor='black')
axes[1].set_title('Average Word Count by Emotion', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Emotion', fontsize=12)
axes[1].set_ylabel('Average Word Count', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Sample Texts by Emotion

In [ ]:
# Display sample texts for each emotion
print("Sample Texts by Emotion:\n")
print("="*80)
for emotion in df['emotion'].unique()[:10]:  # Show first 10 emotions
    print(f"\n{emotion.upper()}:")
    print("-"*80)
    samples = df[df['emotion'] == emotion][text_col].head(3)
    for i, text in enumerate(samples, 1):
        print(f"{i}. {text}")
    print()

## 7. Most Common Words Analysis

In [ ]:
# Get all words from texts
all_words = ' '.join(df[text_col].astype(str)).lower().split()
word_freq = Counter(all_words)
most_common_words = word_freq.most_common(30)

print("Top 30 Most Common Words:")
for word, count in most_common_words:
    print(f"{word}: {count:,}")

In [ ]:
# Visualize most common words
words, counts = zip(*most_common_words[:20])
plt.figure(figsize=(14, 6))
plt.barh(range(len(words)), counts, color='mediumpurple', edgecolor='black')
plt.yticks(range(len(words)), words)
plt.xlabel('Frequency', fontsize=12)
plt.ylabel('Words', fontsize=12)
plt.title('Top 20 Most Common Words', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Unique Words by Emotion

In [ ]:
# Analyze unique vocabulary size per emotion
unique_words_by_emotion = {}
for emotion in df['emotion'].unique():
    emotion_texts = ' '.join(df[df['emotion'] == emotion][text_col].astype(str)).lower()
    unique_words = set(emotion_texts.split())
    unique_words_by_emotion[emotion] = len(unique_words)

unique_words_df = pd.DataFrame(list(unique_words_by_emotion.items()), 
                               columns=['Emotion', 'Unique Words']).sort_values('Unique Words', ascending=False)

print("Unique Vocabulary Size by Emotion:")
print(unique_words_df)

In [ ]:
# Visualize unique words by emotion
plt.figure(figsize=(14, 6))
plt.bar(unique_words_df['Emotion'], unique_words_df['Unique Words'], 
        color='lightcoral', edgecolor='black')
plt.title('Unique Vocabulary Size by Emotion', fontsize=14, fontweight='bold')
plt.xlabel('Emotion', fontsize=12)
plt.ylabel('Number of Unique Words', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Data Quality Checks

In [ ]:
# Check for duplicate texts
duplicate_count = df[text_col].duplicated().sum()
print(f"Number of duplicate texts: {duplicate_count:,}")
print(f"Percentage of duplicates: {(duplicate_count/len(df)*100):.2f}%")

# Check for very short texts
very_short = df[df['word_count'] < 3]
print(f"\nNumber of texts with less than 3 words: {len(very_short):,}")
print(f"Percentage: {(len(very_short)/len(df)*100):.2f}%")

# Check for empty or null texts
empty_texts = df[text_col].isna().sum() + (df[text_col] == '').sum()
print(f"\nNumber of empty texts: {empty_texts}")

## 10. Summary and Key Insights

In [ ]:
print("="*80)
print("DATASET SUMMARY")
print("="*80)
print(f"\n Dataset Overview:")
print(f"   - Total samples: {len(df):,}")
print(f"   - Number of emotions: {df['emotion'].nunique()}")
print(f"   - Features: {', '.join(df.columns.tolist())}")

print(f"\n Text Statistics:")
print(f"   - Average text length: {df['text_length'].mean():.0f} characters")
print(f"   - Average word count: {df['word_count'].mean():.1f} words")
print(f"   - Total unique words: {len(word_freq):,}")

print(f"\n Class Balance:")
print(f"   - Most common: {emotion_counts.index[0]} ({emotion_counts.values[0]:,} samples, {emotion_counts.values[0]/len(df)*100:.1f}%)")
print(f"   - Least common: {emotion_counts.index[-1]} ({emotion_counts.values[-1]:,} samples, {emotion_counts.values[-1]/len(df)*100:.1f}%)")
print(f"   - Imbalance ratio: {emotion_counts.values[0] / emotion_counts.values[-1]:.2f}x")

print(f"\n Data Quality:")
print(f"   - Missing values: {df.isnull().sum().sum()}")
print(f"   - Duplicate texts: {duplicate_count:,} ({duplicate_count/len(df)*100:.2f}%)")
print(f"   - Very short texts (<3 words): {len(very_short):,} ({len(very_short)/len(df)*100:.2f}%)")

print(f"\n Key Insights:")
if emotion_counts.values[0] / emotion_counts.values[-1] > 2:
    print("    Significant class imbalance detected - consider using balancing techniques")
else:
    print("    Classes are relatively balanced")
    
if duplicate_count > len(df) * 0.05:
    print("    High number of duplicate texts - consider deduplication")
else:
    print("    Low duplicate rate")
    
print("\n" + "="*80)

## 11. Save Cleaned Dataset with Features

In [ ]:
# Save dataset with additional features to processed folder
output_path = '../data/processed/emotion_dataset_with_features.csv'
df.to_csv(output_path, index=False)
print(f"✓ Dataset with features saved to: {output_path}")
print(f"  Shape: {df.shape}")
print(f"  Columns: {', '.join(df.columns.tolist())}")